In [1]:
#first let me declare all the necessary packages
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import pybamm
import scipy
from parameter_settings import get_keys_and_values
import time as tm



def run_battery_simulation(a_nmc, b_nmc, c_nmc, d_nmc, my_graphite_diff_parameter, sep_por, neg_por, pos_por, cap_dl_neg, p1, p2, p3, p4, p5, p6, p7, p8):


    def my_graphite_diff(c_s_n,T,diff_param):

        from pybamm import Interpolant,constants, exp
        A_val = 28746 #maximum concentration in negative electrode, mol/m^3
        ratio = c_s_n/A_val #basically, the x in LiC_(6/x) aka Li_xC_6

        x = np.array([0,0.055763963, 0.080379776, 0.101036403, 0.119914264, 0.135205906,
                            0.149751613, 0.158702818, 0.167654023 , 0.183318631, 0.190032035,0.19898324,
                            0.216086435, 0.239263661, 0.257805443, 0.27666691,
                            0.279544083, 0.297446492, 0.312924617, 0.33101351, 0.350194664,
                            0.358986025, 0.378966393, 0.395909745, 0.409336552, 0.430755507,
                            0.438427968, 0.456777938, 0.475191845, 0.497058359, 0.509043028,
                            0.529058916, 0.553994415, 0.576052742, 0.585283671, 0.601787455,
                            0.616333163, 0.630878871, 0.64878128, 0.673397094, 0.699451493,
                            0.72262872, 0.747244533, 0.765146943, 0.776335949, 0.79647616,
                            0.810275934, 0.833399879, 0.843469985, 0.86712674, 0.892701611,
                            0.917317424, 0.941933237, 0.966549051, 0.979975858])

        D = 1e-4*np.array([
            9.62E-12,9.62E-12, 1.05E-11, 1.30E-11, 1.64E-11, 1.64E-11, 1.36E-10, 1.90E-10,2.43E-10,
            3.99E-10, 2.64E-10, 1.68E-10, 7.77E-11, 6.42E-11, 6.55E-11, 7.09E-11, 9.81E-11,
            6.82E-11, 1.39E-10, 1.83E-10, 2.23E-10, 3.36E-10, 2.28E-10, 2.16E-10, 3.62E-10,
            3.86E-10, 5.12E-10, 3.97E-10, 3.40E-10, 3.67E-10, 3.10E-10, 3.25E-10, 3.77E-10,
            3.15E-10, 2.68E-10, 2.29E-10, 4.27E-11, 2.56E-10, 3.58E-10, 3.63E-10, 3.56E-10,
            3.32E-10, 3.57E-10, 3.94E-10, 3.08E-10, 2.99E-10, 4.03E-10, 4.12E-10, 3.58E-10,
            3.34E-10, 3.42E-10, 3.25E-10, 3.29E-10, 3.56E-10, 3.63E-10
        ])




        D_m2s = Interpolant(x,D,ratio,interpolator="cubic") 
        
        return diff_param*D_m2s



    deg_options = {"surface form":"differential", "thermal": "lumped","cell geometry": "arbitrary","SEI":"reaction limited","SEI film resistance":"average","lithium plating":"reversible"} #haven't added mechanical fracture yet 
    default_options = {"thermal":"isothermal"}
    model_SPMe_1 = pybamm.lithium_ion.SPMe(options=deg_options)


    # Get the default solver for the model
    default_solver = model_SPMe_1.default_solver
    print("Default Solver:", default_solver.name)

    var_pts_1 = {
        "x_n": 50,  # negative electrode
        "x_s": 25,  # separator 
        "x_p": 50,  # positive electrode
        "r_n": 50,  # negative particle
        "r_p": 50,  # positive particle
    }

    parameters_1 = pybamm.ParameterValues("Chen2020")

    parameters_1['Cation transference number'] = 0.38
    parameters_1['Cell volume [m3]'] = 3.914e-5

    parameters_1['Electrolyte conductivity [S.m-1]'] = 1.3

    parameters_1['Negative particle radius [m]'] = 2.5e-6

    parameters_1['Positive particle radius [m]'] = 3.5e-6
    parameters_1['Separator porosity'] = sep_por
    parameters_1['Negative electrode porosity'] = neg_por #default value from Chen2020 is 0.25
    parameters_1['Positive electrode porosity'] = pos_por #default value from Chen2020 is 0.335 

    
    keys_and_values = get_keys_and_values(cap_dl_neg)


    parameters_1.update(keys_and_values, check_already_exists=False)

    parameters_1['Cation transference number'] = 0.38
    parameters_1['Cell volume [m3]'] = 3.914e-5
    parameters_1['Electrode height [m]'] = 0.4527
    parameters_1['Electrode width [m]'] = 0.4527
    parameters_1['Electrolyte conductivity [S.m-1]'] = 1.3

    parameters_1['Maximum concentration in negative electrode [mol.m-3]'] = 28746
    parameters_1['Maximum concentration in positive electrode [mol.m-3]'] = 35380


    #use the following to play with SOC ->
    parameters_1['Initial concentration in negative electrode [mol.m-3]'] = 2392.8+11000
    parameters_1['Initial concentration in positive electrode [mol.m-3]'] = 28588-11000
    parameters_1['Nominal cell capacity [A.h]'] = 4.9872 #this doesn't really matter as a parameter - see documentation/github https://github.com/pybamm-team/PyBaMM/discussions/1635#discussioncomment-1261662

    #for positive electrode -

    parameters_1['Positive electrode diffusivity [m2.s-1]'] = 8e-15 #default was 8e-15

    #for negative electrode -
    parameters_1['Negative electrode diffusivity [m2.s-1]'] = lambda c_s_n, T: my_graphite_diff(c_s_n, T, my_graphite_diff_parameter) #default was 5e-15




    def create_custom_experiment_old(p1, p2, p3, p4, p5, p6):

        experiment = pybamm.Experiment([
            ("Rest for 200 seconds", f"Charge at {p1}C for {p2} seconds or until 4.2 V", f"Rest for {p3} seconds", f"Charge at {p4}C for {p5} seconds or until 4.2 V", f"Rest for {p6} seconds", f"Discharge at {p4}C for {p5} seconds or until 3.0 V", "Rest for 300 seconds"),
        ] * 5)
        
        def my_fun(A_values, B_values, omega_values, constant):
                
                def current(t):
                    sine_terms = np.sum([A * pybamm.sin(2 * np.pi * omega * t) for A, omega in zip(A_values, omega_values)], axis=0)
                    cosine_terms = np.sum([B * pybamm.cos(2 * np.pi * omega * t) for B, omega in zip(B_values, omega_values)], axis=0)
                    return sine_terms + cosine_terms + constant
                return current
        

        return experiment

    def create_custom_experiment(p1,p2,p3,p4,p5,p6,p7,p8):


        def my_fun(A, constant):
                
                def current(t):
                    sine_terms = A * pybamm.sin(2 * np.pi * t) 
                    return sine_terms + constant
                return current
        
        t = np.linspace(0, 20, 1000)

        def chirp_signal(f0=0.0,t1=20): 
            chirp = scipy.signal.chirp(t,f0,t1,1)
            return chirp




        drive_cycle_power = np.column_stack([t, chirp_signal(p7,p8)])

        experiment = pybamm.Experiment( [pybamm.step.current(drive_cycle_power)]+[pybamm.step.current(drive_cycle_power),
            ("Rest for 200 seconds", f"Charge at {p1}C for {p2} seconds or until 4.2 V", f"Rest for {p3} seconds", f"Charge at {p4}C for {p5} seconds or until 4.2 V", f"Rest for {p6} seconds", f"Discharge at {p4}C for {p5} seconds or until 3.0 V", "Rest for 300 seconds")
        ] * 5 )
        experiment_ending = my_fun(5,0.1)
        return experiment


    custom_experiment = create_custom_experiment(p1,p2,p3,p4,p5,p6,p7,p8)

    start_time = tm.time()
    experiment = pybamm.Experiment(
        [("Rest for 2 minutes","Charge at 1 C for 12 minutes or until 4.2 V","Rest for 1 hour","Discharge at C/3 for 12 minutes or until 3.0 V","Rest for 1 hour")] 
    )



    initial_voltage = 3.9 
    sim_1 = pybamm.Simulation(model_SPMe_1, experiment=custom_experiment,parameter_values=parameters_1, var_pts=var_pts_1)
    sim_1.solve([0,150])
    end_time = tm.time()
    elapsed_time = end_time - start_time
    print(f"Elapsed time: {elapsed_time:.2f} seconds")
    return sim_1.solution







In [2]:
# Example usage
a_nmc, b_nmc, c_nmc, d_nmc = 153.1353114622245, -299.5867812542784, 182.0307751506729, -65.20873735054764 #uniform for overall diffusion if fine, but what about individual a, b, c, d; gaussian for the overall diffusion  (biggist SD informed by the literature)
my_graphite_diff_parameter = 1 #gaussian is good for figuring out the diffusion - but what if mean if 4 SDs away - is a uniform prior betteR? Could just do a big SD. (SD: 10^-3? Something big that is informed by literature) 
sep_por, neg_por, pos_por = 0.4, 0.25, 0.335 #uniform for porosity is fine; between 20 and 45 percent for porosities; 5 percent error (do we even need to ); 
cap_dl_neg = 0.2 #Will depend on particle size, particle size distributiosn are Gaussian; (SD: averasge value about 10 or 15 microns - a SD might be 5 microns for particle sizes themselves) (so 0.2 mean and 0.1 SD) 



p1 = 0.5  # fraction of C-rate - 0.1 to 2 limit uniform - initial discharge amplitude
p2 = 598.6 # discharge duration - seconds - make sure this is duration of current discharge  - specify as a function of p1 to be between 2% and 10%SOC - change more than 2% of the SOC
p3 = 1800/2  # rest after discharge -  30 minutes in seconds - max 30 minutes, minimum 100 seconds
p4 = 1.0 # HPPC amplitude - fraction of C-rate - HPPC - 0.1 to 2 limit uniform
p5 = 10.0 # HPPC duration - seconds - 1 to 20 seconds 
p6 = 600  # Rest between HPPCs- 10 minutes in seconds - min 10 seconds max 600 seconds 

p7 = 10e-3 #start_frequency - maybe uniform between 1e-3 to 10e-3?
p8 = 20 #end_time - quarter of a period of p7 to full period of p7

# Run the simulation
solution = run_battery_simulation(a_nmc, b_nmc, c_nmc, d_nmc, my_graphite_diff_parameter, sep_por, neg_por, pos_por, cap_dl_neg, p1, p2, p3, p4, p5, p6, p7, p8)


Default Solver: CasADi solver with 'safe' mode


2024-03-15 14:49:15.859 - [WARNING] simulation.solve(602): Ignoring t_eval as solution times are specified by the experiment


Elapsed time: 86.04 seconds


In [3]:
time = solution["Time [s]"].data
terminal_voltage = solution['Terminal voltage [V]'].data